In [ ]:
import numpy as np
import torch
from pathlib import Path
import problem_config as cfg
from pathlib import Path
import matplotlib.pyplot as plt
import pickle
import functools
# my lib
from da_utils import KF_update, EnKF_update, KF_with_sample
from da_utils import PF_resample, ResultRecorder, rk4, lorenz96_drift
from iensf import NoiseSchedule, ScoreRep, ReverseSampler, Langevin, post_score, post_score_static


device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
dtype = cfg.dtype

# path
base_dir   = Path(cfg.__file__).parent.resolve()
data_dir   = base_dir / "problem_data"
result_dir = base_dir / "result"
result_dir.mkdir(exist_ok=True)

# load data
y_np = np.load(data_dir / "obs_path.npy")          # (T+1, m)
y    = torch.tensor(y_np, dtype=dtype, device=device)

x_true_np = np.load(data_dir / "state_path.npy")          # (T+1, m)

# --- Matrices ---
A = cfg.A.to(device=device, dtype=dtype)
H = cfg.H.to(device=device, dtype=dtype)
Q = cfg.Q.to(device=device, dtype=dtype)
R = cfg.R.to(device=device, dtype=dtype)

n = cfg.DIM_STATE
m = cfg.DIM_OBS
T = y.shape[0]



LQ = torch.linalg.cholesky(Q)
LR = torch.linalg.cholesky(R)

In [ ]:

def obs_model(x):
    return x @ H.T

def post_score(zt,t, score_prior, score_likelihood):
    return score_prior(zt,t) + score_likelihood(zt,t)

# --- EnSF params ---
ns = NoiseSchedule(ns_type='linear',eps_a=0.001, eps_b=0.001)
dm = ReverseSampler(noise_schedule=ns)
score_rep = ScoreRep(dm=dm, dim_x=cfg.DIM_STATE, obs_model=obs_model, obs_sigma=cfg.OBS_SIGMA)

# --- EnSF params ---
N_ens = 200
Nt = 100

solver_all = ['SDE_euler', 'ODE_euler', 'DPM_solver']
solver = solver_all[0]



inflation = 1.0
var_split = 1.0
print(f"var_split = {var_split}")
eta1 = 0.4
eta2 = 0.4
N_iter_vec = [1,2,3,4,5]


n_rep = 10

In [ ]:
result_all = {}
for rep_id in range(n_rep):
    print(f"--- Running rep {rep_id+1}/{n_rep} ---")
    for N_iter in N_iter_vec:
        print(f"rep {rep_id} N_iter = {N_iter}")
        
        torch.manual_seed(rep_id)

        # initial ensemble
        m0 = torch.zeros(n, dtype=dtype, device=device)
        P0 = torch.eye(n, dtype=dtype, device=device)
        L0 = torch.linalg.cholesky(P0)
        X  = m0 + torch.randn(N_ens, n, dtype=dtype, device=device) @ L0.T



        # storage
        ms = torch.zeros(T-1, n, dtype=dtype, device=device)
        Ps = torch.zeros(T-1, n, n, dtype=dtype, device=device)


        for k in range(T-1):
            # save results
            ms[k] = X.mean(dim=0)
            Ps[k] = torch.cov(X.T)
            
            X = X @ A.T + torch.randn(N_ens, n, dtype=dtype, device=device) @ LQ.T

            # ---- Analysis ----
            X_cov = torch.cov(X.T)
            X_mean = X.mean(dim=0)
            mu_gmm = inflation*np.sqrt(1-var_split**2) * (X - X_mean) + X_mean
            sigma_gmm = X_cov*var_split            
            
            score_rep.set_obs_value(obs_value=y[k+1])

            x_T = torch.randn(N_ens, n, dtype=dtype, device=device)
            x_T = (x_T - torch.mean(x_T, dim=0))/torch.std(x_T, dim=0)
            
            score_prior_fn = functools.partial(score_rep.score_gaussian_diffusion,
                                                prior_mean=mu_gmm, prior_cov=sigma_gmm)
            current_sample = X.clone()
            mu_star = torch.mean(current_sample, dim=0)
            cov_star = torch.cov(current_sample.T)

            sample_iter = []
            for i in range(N_iter):
                score_l4 = functools.partial(score_rep.score_likelihood_3_full_generic,
                                            prior_mean=mu_gmm, prior_cov=sigma_gmm,
                                            post_mean=mu_star, post_cov=cov_star)

                post_score_fn = functools.partial(post_score, score_prior=score_prior_fn, score_likelihood=score_l4)
                current_sample = dm.sample_gen(x_T=x_T, score_fn=post_score_fn, Nt=Nt, solver_type=solver)
                
                sample_iter.append(current_sample)
                
                current_mean = torch.mean(current_sample, dim=0)
                current_cov = torch.cov(current_sample.T)
                        
                mu_star = (1-eta1)*mu_star + eta1*current_mean
                cov_star = (1-eta2)*cov_star + eta2*current_cov
            
            X = current_sample.clone()
            # # save results
            # ms[k] = current_sample.mean(dim=0)
            # Ps[k] = torch.cov(current_sample.T)

        result_all[f'iensf_{N_iter}_rep{rep_id}'] = (ms.cpu().numpy(), Ps.cpu().numpy())

        # print error
        # print(f"Step {k+1}/{T} \t error = {np.linalg.norm(ms[k].cpu().numpy() - x_true_np[k]):.4f}")    
        # print(X_cov)
        # print(Ps[k])
        

# Save results
out_path = result_dir / "iensf.pkl"
with open(out_path, "wb") as f:
    pickle.dump(result_all, f)
print(f"Saved IEnKF posterior to: {out_path}")


In [ ]:
def kl_gaussian(m0, P0, m1, P1, jitter=1e-9):
    """
    KL( N(m0,P0) || N(m1,P1) ) per step.
    Uses stable solves and adds jitter to keep PD.
    """
    n = P0.shape[-1]
    # jitter for stability
    P0j = P0.copy()
    P1j = P1.copy()
    for M in (P0j, P1j):
        M[...] = 0.5 * (M + M.transpose(0,1))  # symmetrize (if batched later)
    P0j = P0j + jitter * np.eye(n)
    P1j = P1j + jitter * np.eye(n)

    # Cholesky of P1 to get logdet and solve P1^{-1} * X
    L1 = np.linalg.cholesky(P1j)
    logdet_P1 = 2.0 * np.sum(np.log(np.diag(L1)))
    # logdet of P0
    L0 = np.linalg.cholesky(P0j)
    logdet_P0 = 2.0 * np.sum(np.log(np.diag(L0)))

    # trace(P1^{-1} P0) via solves
    # solve L1 * Y = P0  => Y, then solve L1^T * Z = Y => Z = P1^{-1} P0
    Y = np.linalg.solve(L1, P0j)
    Z = np.linalg.solve(L1.T, Y)
    tr_term = np.trace(Z)

    d = (m1 - m0).reshape(-1, 1)  # (n,1)
    # quadratic form d^T P1^{-1} d via solves
    y = np.linalg.solve(L1, d)
    q = y.T @ y

    return float(np.asarray(0.5 * (logdet_P1 - logdet_P0 - n + tr_term + q)).reshape(()))



base_dir = Path(".").resolve()
result_dir = base_dir / "result"
data_dir = base_dir / "problem_data"

# load results
kf = np.load(result_dir / "kf.npz") # ground truth
ms_kf = kf["posterior_mean"]         # (T, n)
Ps_kf = kf["posterior_cov"]          # (T, n, n)
print(f"KF:   {ms_kf.shape}, {Ps_kf.shape}")


# bench mark
with open("result/iensf.pkl", "rb") as f:
    loaded_results = pickle.load(f)


T_total, n = ms_kf.shape
t = np.arange(T_total)


In [ ]:

# compute KL divergence
kl_rep_all = {}
for rep_id in range(n_rep):
    for N_iter in N_iter_vec:
        result_temp = loaded_results[f'iensf_{N_iter}_rep{rep_id}']
        ms, Ps = result_temp
        # compute KL per step
        kl_all = np.zeros(T_total)
        for k in range(T_total):
            m0, P0 = ms_kf[k], Ps_kf[k]
            m, P = ms[k], Ps[k]
            kl_all[k] = kl_gaussian(m0, P0, m, P)
        kl_rep_all[f'iensf_{N_iter}_rep{rep_id}'] = kl_all


In [ ]:
for N_iter in N_iter_vec:
    plt.figure(figsize=(6,3))
    for rep_id in range(n_rep):
        kl_all = kl_rep_all[f'iensf_{N_iter}_rep{rep_id}']
        plt.plot(kl_all, label=f'rep {rep_id}')
    plt.xlabel("time step")
    plt.title(f"KL divergence (iter={N_iter})")
    plt.ylim(0,0.5)
    plt.tight_layout()
    plt.legend()
    plt.show()


In [ ]:
for rep_id in range(n_rep):
    plt.figure(figsize=(6,4))
    for N_iter in N_iter_vec:
        kl_all = kl_rep_all[f'iensf_{N_iter}_rep{rep_id}']
        plt.plot(kl_all, label=f'iter {N_iter}')
    plt.xlabel("time step")
    plt.ylabel("KL divergence")
    plt.ylim(0,0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# stat 
plt.figure(figsize=(6,4))
for N_iter in N_iter_vec:
    kl_all_iter = []
    for rep_id in range(n_rep):
        kl_all = kl_rep_all[f'iensf_{N_iter}_rep{rep_id}']
        kl_all_iter.append(kl_all)
    kl_all_iter = np.array(kl_all_iter)
    kl_all_iter_mean = np.mean(kl_all_iter, axis=0)
    
    # 
    plt.plot(kl_all_iter_mean, label=f'iter {N_iter}')
plt.xlabel("time step")
plt.title(f"KL divergence (iter={N_iter})")
plt.ylim(0,0.5)
plt.tight_layout()
plt.legend()
plt.show()
